# MedFrameQA 论文结果分析

这份 notebook 不跑新实验，只做 5 条方法线的结果汇总：

- `fixed`
- `reasoning`
- `order_vote`
- `order_rerank`
- `order_vote_plus`

它会先调用统一汇总脚本，再读入：

- 最新 high-budget repeat run 对比
- 同协议 repeat-only complete runs
- 按方法聚合的重复运行统计（publication-clean）


In [ ]:
import json
import os
import subprocess
import sys
from pathlib import Path

ROOT = Path("/gluon4/xl693/evolve")
RESULTS_ROOT = ROOT / "results"
OUTPUT_DIR = ROOT / "paper_analysis_output"
BUDGET_ABLATION_ROOT = ROOT / "results_budget_ablation"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MIN_GENERATIONS = 50
METHODS = ["fixed", "reasoning", "order_vote", "order_rerank", "order_vote_plus"]
REPEAT_ONLY = True
SUMMARY_SCRIPT = ROOT / "summarize_medframeqa_paper_runs.py"

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

print("ROOT:", ROOT)
print("RESULTS_ROOT:", RESULTS_ROOT)
print("BUDGET_ABLATION_ROOT:", BUDGET_ABLATION_ROOT)
print("OUTPUT_DIR:", OUTPUT_DIR)
print("MIN_GENERATIONS:", MIN_GENERATIONS)
print("METHODS:", METHODS)
print("REPEAT_ONLY:", REPEAT_ONLY)
print("SUMMARY_SCRIPT:", SUMMARY_SCRIPT)


In [ ]:
cmd = [
    sys.executable,
    str(SUMMARY_SCRIPT),
    "--results-root",
    str(RESULTS_ROOT),
    "--output-dir",
    str(OUTPUT_DIR),
    "--budget-ablation-root",
    str(BUDGET_ABLATION_ROOT),
    "--repeat-only",
    "--methods",
    *METHODS,
]
print("Running summary command:")
print(" ".join(cmd))
subprocess.run(cmd, check=True)


In [ ]:
from medframeqa_runtime import collect_generation_records

latest_rows = json.loads((OUTPUT_DIR / "method_comparison_latest.json").read_text())
all_rows = json.loads((OUTPUT_DIR / "method_comparison_all_runs.json").read_text())
aggregate_rows = json.loads((OUTPUT_DIR / "method_seed_aggregate.json").read_text())
main_table_rows = json.loads((OUTPUT_DIR / "main_table_repeat_only.json").read_text())
budget_sensitivity = json.loads((OUTPUT_DIR / "budget_sensitivity_summary.json").read_text())
decision_summary = json.loads((OUTPUT_DIR / "paper_decision_summary.json").read_text())

latest_by_method = {row["method"]: row for row in latest_rows}
artifacts = []
for row in latest_rows:
    run_dir = Path(row["run_dir"])
    generation_records = collect_generation_records(run_dir)
    artifacts.append(
        {
            "method": row["method"],
            "row": row,
            "run_dir": run_dir,
            "generation_records": generation_records,
        }
    )

print("latest_rows:")
for row in latest_rows:
    print(row)

print("\ndecision_summary:")
print(json.dumps(decision_summary, indent=2, ensure_ascii=False))
print("\nbudget_sensitivity:")
print(json.dumps(budget_sensitivity, indent=2, ensure_ascii=False))


In [ ]:
comparison_rows = latest_rows
print("main_table_rows:")
for row in main_table_rows:
    print(row)

print("\nall aggregate_rows:")
for row in aggregate_rows:
    print(row)


In [ ]:
print("paper decision checks:")
print("- current_main_candidate:", decision_summary["current_main_candidate"])
print("- core_methods_seed_complete:", decision_summary["core_methods_seed_complete"])
print("- reasoning_not_better_than_fixed:", decision_summary["reasoning_not_better_than_fixed"])
print("- order_vote_plus_below_order_rerank:", decision_summary["order_vote_plus_below_order_rerank"])
print("- keep_main_budget_at_50:", decision_summary["keep_main_budget_at_50"])

if not decision_summary["core_methods_seed_complete"]:
    print("\nWARNING: fixed / order_vote / order_rerank 还没有都补到 3 runs，不能作为最终 NeurIPS 主表。")

if budget_sensitivity["available"]:
    print("\nBudget sensitivity summary:")
    print("- 100-gen final_test_score:", budget_sensitivity["final_test_score"])
    print("- 50-gen repeat-only mean final_test_score:", budget_sensitivity["fifty_gen_repeat_only_mean_final_test_score"])
    print("- delta_vs_50gen_mean_final_test:", budget_sensitivity["delta_vs_50gen_mean_final_test"])
    print("- interpretation:", budget_sensitivity["interpretation"])


In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(11, 5))
for artifact in artifacts:
    valid_rows = [
        row
        for row in artifact["generation_records"]
        if row.get("generation") is not None and row.get("combined_score") is not None
    ]
    generations = [row["generation"] for row in valid_rows]
    scores = [row["combined_score"] for row in valid_rows]
    plt.plot(generations, scores, marker="o", linewidth=1.5, label=artifact["method"])

plt.xlabel("Generation")
plt.ylabel("Combined Score")
plt.title("MedFrameQA evolution curves")
plt.grid(alpha=0.25)
plt.legend()
plt.tight_layout()
plt.show()


In [ ]:
final_rows = sorted(comparison_rows, key=lambda row: (-row.get("final_test_score", 0.0), row.get("method")))
print("final test ranking:")
for rank, row in enumerate(final_rows, 1):
    print(
        rank,
        row["method"],
        "| final_test_score =", row["final_test_score"],
        "| selected_holdout_score =", row["selected_holdout_score"],
        "| invalid_rate =", row["invalid_generation_rate"],
    )


In [ ]:
import matplotlib.pyplot as plt

order_vote_main = next(row for row in aggregate_rows if row["method"] == "order_vote")
if budget_sensitivity["available"]:
    labels = ["order_vote 50-gen mean", "order_vote 100-gen single"]
    values = [
        order_vote_main["final_test_score_mean"],
        budget_sensitivity["final_test_score"],
    ]
    plt.figure(figsize=(7, 4))
    bars = plt.bar(labels, values, color=["#2a9d8f", "#e76f51"])
    plt.ylabel("Final Test Score")
    plt.title("Order-Vote Budget Sensitivity")
    plt.ylim(min(values) - 0.03, max(values) + 0.03)
    for bar, value in zip(bars, values):
        plt.text(bar.get_x() + bar.get_width() / 2, value + 0.001, f"{value:.4f}", ha="center")
    plt.tight_layout()
    plt.show()
else:
    print("No budget sensitivity result found.")
